# Classical Image Retrieval Pipeline

This notebook walks through the project step by step: from dataset preparation to feature extraction, ranking, evaluation, and qualitative inspection of the top retrieved images.

The goal is to retrieve images representing the same landmark as the query using classical computer-vision methods only.

We compare two classical representations:
- BoVW (Bag of Visual Words) built from SIFT descriptors
- HOG (Histogram of Oriented Gradients)

For each query, we rank the dataset according to cosine similarity and Euclidean distance, then evaluate with Precision@5 and mAP.

This is a practical notebook version of the execution flow in `scripts/evaluate.py`.

In [ ]:
from __future__ import annotations

import json
import logging
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np

ROOT = Path.cwd()
if str(ROOT) not in os.sys.path:
    os.sys.path.insert(0, str(ROOT))

from src.data.loader import (
    build_class_ground_truth,
    build_query_map,
    ensure_dataset_ready,
    iter_image_paths,
    load_ground_truth,
    load_image,
)
from src.evaluation.metrics import average_precision, mean_average_precision, precision_at_k
from src.features.bovw import build_vocabulary_from_dataset
from src.features.hog import compute_hog_matrix, image_hog
from src.retrieval.search import cosine_rank, euclidean_rank

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

TOP_K = 5
DATASET_ROOT = ROOT / 'data' / 'local'
GT_PATH = DATASET_ROOT / 'ground_truth.json'
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True, parents=True)

print(f'Dataset root: {DATASET_ROOT}')
print(f'Project root: {ROOT}')

## 1) Prepare the dataset and validate the ground truth

The retrieval system needs a dataset of images and a set of query images with relevant matches. In real benchmark datasets, the ground truth is provided explicitly. In this project, the loader first tries to read a ground-truth file and otherwise falls back to a class-based approximation.

This step ensures that the dataset is present and that the queries are valid before feature extraction begins.

In [ ]:
allow_download = os.environ.get('CVIR_ALLOW_DOWNLOAD', '0').strip().lower() in {'1', 'true', 'yes', 'on'}
ensure_dataset_ready(DATASET_ROOT, GT_PATH, allow_download=allow_download)

all_image_paths = sorted(iter_image_paths(DATASET_ROOT))
print(f'Images found: {len(all_image_paths)}')

ground_truth = load_ground_truth(GT_PATH, DATASET_ROOT)
if not ground_truth:
    ground_truth = build_class_ground_truth(DATASET_ROOT)

query_map = build_query_map(DATASET_ROOT)
ground_truth = {
    query_id: [item for item in relevant if (DATASET_ROOT / item).exists()]
    for query_id, relevant in ground_truth.items()
    if query_id in query_map and (DATASET_ROOT / query_id) in all_image_paths
}
ground_truth = {query_id: relevant for query_id, relevant in ground_truth.items() if relevant}

print(f'Evaluable queries: {len(ground_truth)}')
for qid, relevant in list(ground_truth.items())[:3]:
    print(f'{qid} -> {len(relevant)} relevant items')

## 2) Build the BoVW visual vocabulary

Bag of Visual Words (BoVW) turns local image descriptors into a compact histogram. The idea is: extract SIFT descriptors from many images, cluster them with k-means, and treat each cluster center as a visual word.

This is analogous to building a visual dictionary from image patches. Once the vocabulary is learned, each image becomes a histogram over these words.

The notebook uses a bounded sampling strategy so the vocabulary can be trained efficiently without processing every descriptor in the full dataset.

In [ ]:
vocabulary_size = 96
bovw_model = build_vocabulary_from_dataset(DATASET_ROOT, vocabulary_size=vocabulary_size)
print(f'BoVW vocabulary trained with {bovw_model.vocabulary_size} visual words')

sample_query = next(iter(ground_truth))
sample_hist = bovw_model.image_histogram(DATASET_ROOT / sample_query)
print(f'Query histogram shape: {sample_hist.shape}')
print(f'Histogram sum: {sample_hist.sum():.4f}')

## 3) Compute BoVW histograms for the database and queries

Each image is represented by a fixed-length vector whose dimensions equal the vocabulary size. The histogram counts how many local descriptors are assigned to each visual word.

Normalizing this vector makes it comparable across images with different numbers of detected keypoints. The resulting representation is compact and fast to compare with similarity metrics.

In [ ]:
bovw_matrix, bovw_ids = bovw_model.compute_database_histograms(all_image_paths, DATASET_ROOT)
bovw_query_vectors = {
    qid: bovw_model.image_histogram(DATASET_ROOT / qid) for qid in ground_truth
}

print(f'BoVW matrix shape: {bovw_matrix.shape}')
print(f'First database IDs: {bovw_ids[:5]}')
print(f'First query image: {next(iter(bovw_query_vectors))}')

## 4) Compute HOG descriptors as a global baseline

HOG captures the distribution of gradient orientations in image regions. Unlike SIFT, HOG is a global descriptor; it is not based on keypoints but on a dense local image structure.

This gives us a strong classical counterpoint to BoVW: one method is local and vocabulary-based, while the other is dense and global. Comparing them helps reveal which representation is more suitable for landmark retrieval.

In [ ]:
hog_matrix, hog_ids = compute_hog_matrix(all_image_paths, DATASET_ROOT)
hog_query_vectors = {qid: image_hog(DATASET_ROOT / qid) for qid in ground_truth}

print(f'HOG matrix shape: {hog_matrix.shape}')
print(f'HOG IDs sample: {hog_ids[:5]}')
print(f'HOG query vector length: {len(hog_query_vectors[next(iter(ground_truth))])}')

## 5) Rank the database for each query

Once each image has a fixed-length feature vector, we can compare the query to every dataset image. We use two standard similarity measures:

- cosine similarity: measures angular agreement between vectors
- Euclidean distance: measures absolute geometric distance in feature space

The retrieval pipeline sorts the dataset by similarity and returns the top-k most similar items. The ranking is what we later evaluate against the known relevant ground-truth matches.

In [ ]:
def rank_query(query_id: str, query_vector, feature_matrix, ids: list[str], metric: str, top_k: int = 5):
    if metric == 'cosine':
        order = cosine_rank(query_vector, feature_matrix, top_k=None)
    elif metric == 'euclidean':
        order = euclidean_rank(query_vector, feature_matrix, top_k=None)
    else:
        raise ValueError(f'Unsupported metric: {metric}')

    ranked = [ids[idx] for idx in order if ids[idx] != query_id]
    return ranked[:top_k]

metric_names = ['cosine', 'euclidean']

bovw_rankings = {
    metric: {qid: rank_query(qid, bovw_query_vectors[qid], bovw_matrix, bovw_ids, metric, TOP_K) for qid in ground_truth}
    for metric in metric_names
}

hog_rankings = {
    metric: {qid: rank_query(qid, hog_query_vectors[qid], hog_matrix, hog_ids, metric, TOP_K) for qid in ground_truth}
    for metric in metric_names
}

for metric in metric_names:
    sample_qid = next(iter(ground_truth))
    print(f'{metric} ranking for {sample_qid}:')
    print(bovw_rankings[metric][sample_qid][:5] if metric == 'cosine' else hog_rankings[metric][sample_qid][:5])
    print()

## 6) Evaluate retrieval quality

Evaluation compares the retrieved list against ground truth. A good retrieval system should put relevant images near the top.

We measure:
- Precision@5: fraction of top-5 results that are relevant
- AP (Average Precision): summarizes the quality of the ranking across all relevant images
- mAP: the mean AP across all queries

These metrics tell us whether the system retrieves the correct landmark and whether it ranks those matches early.

In [ ]:
def evaluate_retrieval(relevant_map, ranking_map, top_k=5):
    metrics = {}
    for metric_name in ['cosine', 'euclidean']:
        ranked_lists = ranking_map[metric_name]
        query_relevant = {qid: set(relevant) for qid, relevant in relevant_map.items() if qid in ranked_lists}
        per_q_p5 = {
            qid: precision_at_k(set(relevant), ranked_lists[qid], top_k)
            for qid, relevant in query_relevant.items()
        }
        ap_scores = {
            qid: average_precision(set(relevant), ranked_lists[qid])
            for qid, relevant in query_relevant.items()
        }
        metrics[metric_name] = {
            'mean_precision_at_5': sum(per_q_p5.values()) / max(len(per_q_p5), 1),
            'mAP': mean_average_precision(query_relevant, ranked_lists),
            'precision_at_5_by_query': per_q_p5,
            'AP_by_query': ap_scores,
        }
    return metrics

bovw_metrics = evaluate_retrieval(ground_truth, bovw_rankings)
hog_metrics = evaluate_retrieval(ground_truth, hog_rankings)

print('BoVW metrics:')
for metric_name, vals in bovw_metrics.items():
    print(metric_name, vals['mAP'], vals['mean_precision_at_5'])

print('\nHOG metrics:')
for metric_name, vals in hog_metrics.items():
    print(metric_name, vals['mAP'], vals['mean_precision_at_5'])

## 7) Visualize the top-5 retrieved results

The numbers tell us whether the retrieval is good on average, but they do not show why a result succeeds or fails. Qualitative inspection is essential in image retrieval.

For a few query images, we display the query together with the top five results. This helps us understand:
- whether the model retrieves the same building
- whether viewpoint or lighting changes cause confusion
- whether the representation is too local or too global for a given query

This is the visual context that complements the evaluation metrics.

In [ ]:
def show_top5_panel(query_id: str, ranked_results: list[str], dataset_root: Path, title: str = ''):
    fig, axes = plt.subplots(1, TOP_K + 1, figsize=(14, 3.2))
    paths = [dataset_root / query_id] + [dataset_root / item for item in ranked_results[:TOP_K]]
    labels = ['query'] + [f'top {i}' for i in range(1, TOP_K + 1)]

    for ax, path, label in zip(axes, paths, labels):
        image = load_image(path)
        ax.imshow(image)
        ax.set_title(label, fontsize=9)
        ax.axis('off')

    fig.suptitle(title or f'Query: {query_id}', fontsize=12)
    fig.tight_layout()
    plt.show()

selected_queries = list(ground_truth.keys())[:4]
for qid in selected_queries:
    ranked = bovw_rankings['cosine'][qid][:TOP_K]
    show_top5_panel(qid, ranked, DATASET_ROOT, title=f'BoVW cosine retrieval for {qid}')

## 8) Interpretation and discussion

The final step is not to simply report numbers. It is to interpret what they mean:

- A high mAP means the correct images appear early in the ranking
- A low Precision@5 indicates the system is retrieving many incorrect landmarks
- A strong top-5 comparison image is a sign that the representation captures the landmark's visual structure
- Failure cases often come from viewpoint change, repeated architecture, or similar building facades

This is also where you connect the visual evidence to the representation itself: BoVW depends on local keypoints and visual-word assignment, while HOG summarizes overall structure and gradients. For landmark retrieval, both can be useful but often fail under extreme viewpoint changes or visually similar buildings.

In [ ]:
summary = {
    'bovw': bovw_metrics,
    'hog': hog_metrics,
    'dataset_images': len(all_image_paths),
    'evaluated_queries': len(ground_truth),
    'top_k': TOP_K,
}

print(json.dumps({
    'dataset_images': summary['dataset_images'],
    'evaluated_queries': summary['evaluated_queries'],
    'bovw_cosine_mAP': summary['bovw']['cosine']['mAP'],
    'bovw_euclidean_mAP': summary['bovw']['euclidean']['mAP'],
    'hog_cosine_mAP': summary['hog']['cosine']['mAP'],
    'hog_euclidean_mAP': summary['hog']['euclidean']['mAP'],
}, indent=2))